# Instalacja i importy

In [2]:
import numpy as np
import random
from collections import defaultdict
from tqdm import trange

# Środowisko gry

In [3]:
class TicTacToe:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [" "] * 9
        self.current_player = random.choice(["X", "O"])  # losowo kto zaczyna
        return tuple(self.board)

    def available_moves(self):
        return [i for i, x in enumerate(self.board) if x == " "]

    def step(self, action):
        self.board[action] = self.current_player
        winner = self.check_winner()
        done = winner is not None or " " not in self.board

        rewards = {"X": 0, "O": 0}
        if done:
            if winner == "X":
                rewards = {"X": 1, "O": -1}
            elif winner == "O":
                rewards = {"X": -1, "O": 1}
            else:  # remis
                rewards = {"X": 0, "O": 0}

        self.current_player = "O" if self.current_player == "X" else "X"
        return tuple(self.board), rewards, done

    def check_winner(self):
        lines = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),  # poziome
            (0, 3, 6), (1, 4, 7), (2, 5, 8),  # pionowe
            (0, 4, 8), (2, 4, 6)              # ukośne
        ]
        for a, b, c in lines:
            if self.board[a] == self.board[b] == self.board[c] != " ":
                return self.board[a]
        return None

# Q-learning

In [4]:
class QLearningAgent:
    def __init__(self, symbol, alpha=0.1, gamma=0.9, epsilon=0.8):
        self.symbol = symbol
        self.q_table = defaultdict(float)
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q_table[(state, action)]

    def choose_action(self, state, available_moves, exploit=False):
        if not exploit and random.random() < self.epsilon:
            return random.choice(available_moves)
        qs = [self.get_q(state, a) for a in available_moves]
        max_q = max(qs)
        best_moves = [a for a, q in zip(available_moves, qs) if q == max_q]
        return random.choice(best_moves)

    def learn(self, state, action, reward, next_state, next_moves):
        old_q = self.q_table[(state, action)]
        if next_moves:
            next_q = max([self.get_q(next_state, a) for a in next_moves])
        else:
            next_q = 0
        self.q_table[(state, action)] = old_q + self.alpha * (reward + self.gamma * next_q - old_q)


# Trening agenta

In [5]:
def train(agent_x, episodes=50000, decay=0.9999):
    for ep in trange(episodes, desc="Trening"):
        env = TicTacToe()
        state = env.reset()
        done = False

        while not done:
            if env.current_player == "X":
                moves = env.available_moves()
                action = agent_x.choose_action(state, moves)
                next_state, rewards, done = env.step(action)
                next_moves = env.available_moves()
                agent_x.learn(state, action, rewards["X"], next_state, next_moves)
                state = next_state
            else:
                moves = env.available_moves()
                action = random.choice(moves)  # losowy O
                state, rewards, done = env.step(action)

        agent_x.epsilon *= decay  # zmniejszamy epsilon

# Testowanie skuteczności

In [6]:
def test(agent_x, episodes=50000):
    results = {"X_win": 0, "O_win": 0, "draw": 0}

    for _ in trange(episodes, desc="Testowanie"):
        env = TicTacToe()
        state = env.reset()
        done = False

        while not done:
            if env.current_player == "X":
                moves = env.available_moves()
                action = agent_x.choose_action(state, moves, exploit=True)
                state, rewards, done = env.step(action)
            else:
                moves = env.available_moves()
                action = random.choice(moves)
                state, rewards, done = env.step(action)

        winner = env.check_winner()
        if winner == "X":
            results["X_win"] += 1
        elif winner == "O":
            results["O_win"] += 1
        else:
            results["draw"] += 1

    total = sum(results.values())
    print(f"\n📊 Wyniki z {total} gier:")
    print(f"X wygrane : {100*results['X_win']/total:.2f}%")
    print(f"O wygrane : {100*results['O_win']/total:.2f}%")
    print(f"Remisy    : {100*results['draw']/total:.2f}%")
    print(f"X przegrane: {100*results['O_win']/total:.2f}%")

# Uruchomienie

In [7]:

training_episodes_list = [1000, 10000, 100000, 1000000]

for episodes in training_episodes_list:
    print(f"\n=== Trening agenta X na {episodes} epizodach ===")
    agent_x = QLearningAgent("X")  # nowy agent dla każdego treningu
    train(agent_x, episodes=episodes)
    test(agent_x, episodes=100000)  # test na 50k gier


=== Trening agenta X na 1000 epizodach ===


Testowanie: 100%|██████████| 100000/100000 [00:02<00:00, 43627.42it/s]



📊 Wyniki z 100000 gier:
X wygrane : 47.29%
O wygrane : 40.72%
Remisy    : 12.00%
X przegrane: 40.72%

=== Trening agenta X na 10000 epizodach ===


Testowanie: 100%|██████████| 100000/100000 [00:02<00:00, 46855.58it/s]



📊 Wyniki z 100000 gier:
X wygrane : 68.05%
O wygrane : 25.47%
Remisy    : 6.48%
X przegrane: 25.47%

=== Trening agenta X na 100000 epizodach ===


Testowanie: 100%|██████████| 100000/100000 [00:02<00:00, 47953.59it/s]



📊 Wyniki z 100000 gier:
X wygrane : 76.30%
O wygrane : 18.86%
Remisy    : 4.84%
X przegrane: 18.86%

=== Trening agenta X na 1000000 epizodach ===


Testowanie: 100%|██████████| 100000/100000 [00:02<00:00, 46872.45it/s]


📊 Wyniki z 100000 gier:
X wygrane : 74.71%
O wygrane : 20.32%
Remisy    : 4.97%
X przegrane: 20.32%
